# Paths to Causality — Demo trên Google Colab (T4 GPU)

Chạy toàn bộ demo tự host (Streamlit + Neo4j + Hetionet + Ollama) trên một máy Colab,
tận dụng **T4 GPU** để suy luận cross-encoder (RoBERTa) đã huấn luyện ở Module 2.

**Chọn runtime trước khi chạy:** `Runtime → Change runtime type → T4 GPU`.

Luồng notebook:
1. Clone repo + cài dependencies (torch/transformers có sẵn trên Colab).
2. Tải weights đã huấn luyện từ Google Drive (gdown) và giải nén về **đường dẫn cố định** của demo.
3. Cài Neo4j standalone + nạp Hetionet v1.0 (47.031 nodes / 2.250.197 edges).
4. Cài Ollama + pull `qwen2.5:7b` (LLM local cho pipeline).
5. Khởi động Streamlit và truy cập UI qua tunnel Cloudflare.
6. Smoke-test: cross-encoder chạy trên GPU, ranker chain, và một lần `analyze_pair` end-to-end.

## 1. Clone repo + cài dependencies

In [ ]:
%cd /content
!rm -rf Reproduce_Paths-to-Causality_KDD-25
!git clone -b demo https://github.com/KhangNLY/Reproduce_Paths-to-Causality_KDD-25.git
%cd /content/Reproduce_Paths-to-Causality_KDD-25/demo

In [ ]:
# Dependencies của demo. torch/transformers Colab đã có sẵn (bỏ comment 2 dòng dưới nếu thiếu).
!pip -q install gdown streamlit neo4j xgboost pyvis python-dotenv cloudscraper 2>/dev/null | tail -1
# !pip -q install torch transformers

import importlib
for m in ("torch", "transformers", "streamlit", "neo4j", "xgboost"):
    ok = importlib.util.find_spec(m) is not None
    print(f"  {m:12} {'OK' if ok else 'THIẾU'}")
import torch
print("CUDA:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "chỉ CPU")

## 2. Tải weights đã huấn luyện từ Google Drive
5 bộ weights: 4 checkpoint cross-encoder RoBERTa (ADE / COMAGC / GENEC / SemEval) và 1 bộ XGBoost + n-gram (all tasks).

In [ ]:
%cd /content/Reproduce_Paths-to-Causality_KDD-25
!mkdir -p saved_models

# 1. ptc_weights_cross_encoder_ade_roberta_best.zip
!gdown 1UOTaEeB8yv10WjsqYFtETYnNSgJpc-vo -O saved_models/ade_roberta.zip
# 2. ptc_weights_cross_encoder_comagc_roberta_best.zip
!gdown 1NDmPTK68DgiMZWCHYjkodQtqY_IGKjQp -O saved_models/comagc_roberta.zip
# 3. ptc_weights_cross_encoder_gene_roberta_best.zip
!gdown 1AksCE7gRf5IUKssJ1me6WmYNuSSxtF13 -O saved_models/gene_roberta.zip
# 4. ptc_weights_cross_encoder_semeval_roberta_best.zip
!gdown 1ul9J3RsbZri6TlPRgc0lztXhk0XXHYE1 -O saved_models/semeval_roberta.zip
# 5. ptc_weights_xgboost_all_tasks_best_20261104.zip
!gdown 1QHj-z_yAGC8DTEUgmT3hnDKNjmZFIdWg -O saved_models/xgboost_all_tasks.zip

## 3. Giải nén weights về đường dẫn cố định của demo
Demo tìm checkpoint tại **đường dẫn cố định** (xem README mục *Model checkpoints: fixed locations*):
- Cross-encoder: `demo/models/ce/model_best_<task>_<ranker>_<ts>/` (mỗi thư mục đủ `model.safetensors` + tokenizer).
- XGBoost + n-gram: `demo/models/repo_ranker/` (`xgb_ranker.json` + `ngram_embeddings.npz` + `config.json`).

Cell dưới giải nén zip, dò mọi thư mục có `model.safetensors` và copy về đúng vị trí.

In [ ]:
%cd /content/Reproduce_Paths-to-Causality_KDD-25
import zipfile, os, shutil, json
from pathlib import Path

saved_models_dir = "/content/Reproduce_Paths-to-Causality_KDD-25/saved_models"
for file in os.listdir(saved_models_dir):
    if file.endswith(".zip"):
        file_path = os.path.join(saved_models_dir, file)
        extract_path = os.path.join(saved_models_dir, file.replace(".zip", ""))
        print(f"Đang giải nén {file} vào {extract_path}...")
        with zipfile.ZipFile(file_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
print("=== Giải nén hoàn tất!")
!ls -lh saved_models

In [ ]:
# Dò kết quả giải nén và sắp vào ĐÚNG đường dẫn cố định của demo.
root = Path("/content/Reproduce_Paths-to-Causality_KDD-25")
ce_root = root / "demo" / "models" / "ce"
ce_root.mkdir(parents=True, exist_ok=True)

# (a) Cross-encoder: mọi thư mục chứa model.safetensors -> demo/models/ce/<tên gốc>
ce_dirs = [d for d in Path(saved_models_dir).rglob("model.safetensors")]
print(f"Thấy {len(ce_dirs)} checkpoint cross-encoder")
for f in ce_dirs:
    src = f.parent
    dst = ce_root / src.name
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    meta = {}
    mp = dst / "train_meta.json"
    if mp.exists():
        meta = json.loads(mp.read_text())
    print(f"  {src.name}  (ranker={meta.get('ranker','?')}, task={meta.get('task','?')}, "
          f"NDCG@5={meta.get('test_corrected_top5_NDCG@5','?')})")

# (b) XGBoost + n-gram: bộ 3 file -> demo/models/repo_ranker
xgb_dst = root / "demo" / "models" / "repo_ranker"
cfgs = [p.parent for p in Path(saved_models_dir).rglob("xgb_ranker.json")]
if cfgs:
    # nếu zip có nhiều task, gộp: chọn thư mục mới nhất theo tên
    src = sorted(cfgs, key=lambda p: p.name)[-1]
    xgb_dst.mkdir(parents=True, exist_ok=True)
    for name in ("xgb_ranker.json", "ngram_embeddings.npz", "config.json"):
        if (src / name).exists():
            shutil.copy2(src / name, xgb_dst / name)
    print(f"XGBoost artifact: {src} -> {xgb_dst}")
else:
    print("Không thấy artifact XGBoost trong zip (demo vẫn chạy với CE/local XGB).")

!ls -R /content/Reproduce_Paths-to-Causality_KDD-25/demo/models | head -30

## 4. Cài Neo4j standalone + nạp Hetionet
Không dùng Docker trên Colab — cài bản Neo4j Community trực tiếp, tạo password rồi import Hetionet (~2 phút).

In [ ]:
# Cài Neo4j 5 community (standalone, chạy trong VM của Colab) — an toàn khi chạy lại
import os, pathlib, subprocess, time

if not pathlib.Path("/opt/neo4j/bin/neo4j").exists():
    !wget -q https://neo4j.com/artifact.php?name=neo4j-community-5.26.0-unix.tar.gz -O /tmp/neo4j.tar.gz
    !tar -xzf /tmp/neo4j.tar.gz -C /opt
    !mv /opt/neo4j-community-5.26.0 /opt/neo4j
else:
    print("Neo4j đã cài, bỏ qua bước cài lại.")

# Cho phép binding + GIỚI HẠN BỘ NHỚ cho vừa RAM ~12GB của Colab
# (mặc định Neo4j xin nhiều hơn -> process dễ bị OOM kill ngay sau khi start).
conf = pathlib.Path("/opt/neo4j/conf/neo4j.conf")
txt = conf.read_text()
txt = txt.replace("#server.default_listen_address=0.0.0.0", "server.default_listen_address=0.0.0.0")
lines = [l for l in txt.splitlines()
         if not l.startswith(("server.memory.heap.initial_size",
                              "server.memory.heap.max_size",
                              "server.memory.pagecache.size"))]
conf.write_text("\n".join(lines) +
    "\nserver.memory.heap.initial_size=512m"
    "\nserver.memory.heap.max_size=1G"
    "\nserver.memory.pagecache.size=1G\n")

# Password theo mặc định .env của demo (chỉ có tác dụng trước lần start đầu)
if not pathlib.Path("/opt/neo4j/data/dbms/auth.ini").exists():
    !/opt/neo4j/bin/neo4j-admin dbms set-initial-password local-demo-change-me

# Start DETACHED (session riêng): Colab giết process nền của cell đã kết thúc,
# neo4j start qua !shell sẽ chết theo cell -> dùng console + start_new_session.
_log = open("/tmp/neo4j-console.log", "w")
subprocess.Popen(["/opt/neo4j/bin/neo4j", "console"],
                 start_new_session=True, stdout=_log, stderr=subprocess.STDOUT,
                 cwd="/opt/neo4j")

# Lần start đầu phải khởi tạo DB — có thể mất 40-90s. Poll thay vì sleep cố định.
ready = False
for i in range(36):  # tối đa 3 phút
    time.sleep(5)
    r = subprocess.run(
        ["/opt/neo4j/bin/cypher-shell", "-u", "neo4j", "-p", "local-demo-change-me", "RETURN 1 AS ok"],
        capture_output=True, text=True)
    if r.returncode == 0:
        ready = True
        print(f"Neo4j sẵn sàng sau ~{(i+1)*5}s"); break
    print(f"đang chờ Neo4j... ({(i+1)*5}s)")
if not ready:
    print("CHƯA xong sau 3 phút — 12 dòng log cuối để chẩn đoán:")
    print(subprocess.run("tail -12 /opt/neo4j/logs/neo4j.log", shell=True,
                         capture_output=True, text=True).stdout)
else:
    print(r.stdout)

In [ ]:
# Tải Hetionet v1.0 về demo/data/raw (script chính của repo) rồi import vào Neo4j.
# Trỏ NEO4J_URI sang localhost vì không có Docker network.
%cd /content/Reproduce_Paths-to-Causality_KDD-25/demo
import os
os.environ["NEO4J_URI"] = "bolt://localhost:7687"

!bash scripts/download_hetionet.sh
!python scripts/load_hetionet.py
! /opt/neo4j/bin/cypher-shell -u neo4j -p local-demo-change-me "MATCH (n:Entity) RETURN count(n) AS nodes" 

## 5. Cài Ollama + pull `qwen2.5:7b`
LLM local cho bước phân loại + dịch. Trên Colab, Ollama chạy server trong VM, model ~4.7 GB — lưu ý T4 có 16 GB VRAM, RoBERTa (~0.5 GB) + Qwen 7B Q4 (~5 GB) vẫn thoải mái.

In [ ]:
# Image Colab thiếu zstd — script cài Ollama cần nó để giải nén.
!apt-get install -y -q zstd
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
# start DETACHED: sống qua nhiều cell (Colab giết process nền của cell đã xong)
proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                 start_new_session=True)  # sống qua nhiều cell
time.sleep(8)
!ollama pull qwen2.5:7b
!ollama list

## 6. Khởi động Streamlit + tunnel Cloudflare
Cell dưới chạy Streamlit (port 8501) và cloudflared, in ra **đường link public** (`https://xxx.trycloudflare.com`) — mở link đó để dùng demo từ trình duyệt của bạn.

In [ ]:
%cd /content/Reproduce_Paths-to-Causality_KDD-25/demo

# .env cho môi trường Colab: Neo4j localhost, Ollama localhost, bật CE (torch đã có)
env = """NEO4J_URI=bolt://localhost:7687
NEO4J_USER=neo4j
NEO4J_PASSWORD=local-demo-change-me
OLLAMA_BASE_URL=http://localhost:11434
OLLAMA_MODEL=qwen2.5:7b
MAX_HOPS=4
MAX_PATHS=30
TOP_K_PATHS=3
RANKER_PATH=models/xgb_ranker.json
REPO_CE_DIR=models/ce
REPO_RANKER_DIR=models/repo_ranker
"""
open(".env", "w").write(env)

# cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Dọn process cũ chiếm cổng (chạy lại cell nhiều lần không làm kẹt 8501)
subprocess.run("pkill -f streamlit; pkill -f cloudflared; sleep 2", shell=True)
subprocess.run("fuser -k 8501/tcp 2>/dev/null", shell=True)
time.sleep(3)

import subprocess, time, re, requests, socket
subprocess.Popen(["streamlit", "run", "app.py", "--server.port=8501", "--server.headless=true"],
                 cwd="/content/Reproduce_Paths-to-Causality_KDD-25/demo",
                 start_new_session=True,
                 stdout=open("/tmp/streamlit.log", "w"), stderr=subprocess.STDOUT)
# Chờ app THẬT SỰ lắng nghe (tránh tunnel trỏ vào cổng trống)
for i in range(30):
    time.sleep(2)
    try:
        socket.create_connection(("localhost", 8501), timeout=2).close()
        print(f"Streamlit lắng nghe cổng 8501 sau ~{(i+1)*2}s"); break
    except OSError:
        pass
else:
    print("Streamlit chưa lên — log:"); print(open("/tmp/streamlit.log").read()[-1500:])
tunnel = subprocess.Popen(["cloudflared", "tool", "--url", "http://localhost:8501"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                          start_new_session=True)
url = None
for _ in range(60):
    line = tunnel.stdout.readline()
    if not line: break
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m: url = m.group(0); break
print("UI demo:", url or "(xem log tunnel ở cell sau)")
if url is None:
    # đọc tiếp vài giây
    for _ in range(30):
        line = tunnel.stdout.readline()
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line or "")
        if m: print("UI demo:", m.group(0)); break

## 7. Smoke-test trên T4
Kiểm tra: cross-encoder load lên **GPU**, chuỗi ranker đúng thứ tự ưu tiên, và một lần suy luận `analyze_pair` end-to-end (Hetionet → CE → Ollama).

In [ ]:
%cd /content/Reproduce_Paths-to-Causality_KDD-25/demo
import sys, os, json, time
sys.path.insert(0, ".")
os.environ.setdefault("NEO4J_URI", "bolt://localhost:7687")

import torch
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

from src.repo_ce_ranker import discover_best_ce, RepoCrossEncoderRanker
best = discover_best_ce("models/ce")
print("Checkpoint tốt nhất:", best)
t0 = time.time()
ce = RepoCrossEncoderRanker.try_load(best)
print(f"Load {time.time()-t0:.1f}s | device={ce.device}")
paths = [
    {"path_nodes": [{"name": "TRPV6", "kind": "Gene"}, {"name": "TRAF1", "kind": "Gene"}, {"name": "prostate cancer", "kind": "Disease"}]},
    {"path_nodes": [{"name": "FGF6", "kind": "Gene"}, {"name": "prostate cancer", "kind": "Disease"}]},
]
t0 = time.time()
scores = ce.score("FGF6", "prostate cancer", paths * 8)
print(f"CE inference: {len(paths)*8} paths / {time.time()-t0:.2f}s ({(time.time()-t0)/(len(paths)*8)*1000:.0f} ms/path)")

from src.ranker import PathRanker
pr = PathRanker()
print("PathRanker mode:", pr.mode)
for v, lbl in PathRanker.choices():
    print("  option:", v, "|", lbl[:70])

In [ ]:
# End-to-end: chọn sample COMAGC thật từ benchmark và chạy 1 lần analyze_pair
row = json.loads(open("data/benchmark/comagc/processed_test_truth.jsonl", encoding="utf-8").readline())
sample = dict(row); sample["dataset"] = "comagc"; sample["source_file"] = "test_truth.jsonl"
e1 = {"id": "x", "name": row["e1"], "kind": "Gene"}
e2 = {"id": "y", "name": row["e2"], "kind": "Disease"}
print("Cặp:", row["e1"], "→", row["e2"])

from src.pipeline import analyze_pair
r = analyze_pair(e1, e2, row["sentence"][:200], mode="paper", sample=sample)
print("path_source:", r["path_source"], "| paths:", r["paths_found"])
print("ranker_mode:", r["ranker_mode"])
print("prediction:", r["prediction"]["label"])

## Ghi chú
- **Link tunnel** chỉ sống trong phiên Colab; restart runtime thì chạy lại cell 6 để lấy link mới (các bước 1–5 không cần làm lại nếu VM còn).
- Trên UI: sidebar **Ranking model** giờ liệt kê đầy đủ 4 checkpoint cross-encoder (ADE/COMAGC/GENEC/SemEval) kèm NDCG@5 — chọn bất kỳ để chạy trên T4; **Pipeline LLM** có thể đổi sang OpenAI-compatible (OpenRouter…) nếu muốn LLM mạnh hơn qwen2.5:7b.
- RAM Colab ~12 GB: Neo4j heap mặc định thấp, đủ cho Hetionet; nếu `MemoryError` khi import, thêm `server.memory.heap.max.size=2G` vào `/opt/neo4j/conf/neo4j.conf` rồi `!/opt/neo4j/bin/neo4j restart`.
- Ground truth không được gửi vào prompt; nhãn causal/non-causal chỉ là dự đoán của model.